# Notebook 34 — Workflow Versioning

Safe iteration in production: every change is tracked, diffs are instant, rollbacks are non-destructive.

| Class | Purpose |
|---|---|
| `WorkflowVersion` | immutable snapshot of a workflow definition |
| `VersionedWorkflow` | workflow with full version history |
| `WorkflowDiff` | structured diff between two versions |
| `InMemoryVersionStore` | ephemeral store |
| `SQLiteVersionStore` | durable SQLite store |
| `ChangeLog` | human-readable history |

## 1. Setup

In [ ]:
import tempfile
from multigen.versioning import (
    WorkflowVersion, VersionedWorkflow, WorkflowDiff,
    InMemoryVersionStore, SQLiteVersionStore, ChangeLog,
)

tmpdir = tempfile.mkdtemp()
store  = SQLiteVersionStore(f'{tmpdir}/wf.db')
wf     = VersionedWorkflow('report-pipeline', store=store)
print('VersionedWorkflow ready')

## 2. Commit versions

In [ ]:
# Initial version
v1 = await wf.commit(
    definition={
        'steps': ['load_data', 'clean_data', 'analyse', 'generate_report'],
        'timeout': 3600,
        'retry': 0,
    },
    message='Initial pipeline',
    author='alice',
)
print(f'v1: {v1}')

# Add validation step
v2 = await wf.commit(
    definition={
        'steps': ['load_data', 'validate_schema', 'clean_data', 'analyse', 'generate_report'],
        'timeout': 3600,
        'retry': 1,
    },
    message='Add schema validation + retry',
    author='bob',
    tags=['staging'],
)
print(f'v2: {v2}')

# Performance optimisation
v3 = await wf.commit(
    definition={
        'steps': ['load_data', 'validate_schema', 'clean_data', 'analyse', 'cache_results', 'generate_report'],
        'timeout': 1800,
        'retry': 1,
        'parallel': True,
    },
    message='Add caching, halve timeout, enable parallel',
    author='charlie',
    tags=['staging', 'performance'],
)
print(f'v3: {v3}')

## 3. Diff between versions

In [ ]:
diff_12 = wf.diff(v1.version, v2.version)
print(diff_12.summary())
print()
diff_23 = wf.diff(v2.version, v3.version)
print(diff_23.summary())

## 4. Changelog

In [ ]:
cl = await wf.changelog()
print(cl.render())

## 5. Tag stable versions

In [ ]:
await wf.tag(v2.version, 'stable')
tagged = await store.get('report-pipeline', v2.version)
print(f'{v2.version} tags: {tagged.tags}')

## 6. Rollback (non-destructive)

In [ ]:
# Bug found in v3 — roll back to v1
rollback_v = await wf.rollback(v1.version)
print(f'Rollback created: {rollback_v}')
print(f'Rollback definition = v1 definition: {rollback_v.definition == v1.definition}')

# History preserved
history = await wf.history()
print(f'History ({len(history)} versions): {[h.version for h in history]}')

## 7. Multiple workflows

In [ ]:
wf2 = VersionedWorkflow('data-ingestion', store=store)
await wf2.commit({'source': 's3', 'format': 'parquet'}, message='init')
await wf2.commit({'source': 's3', 'format': 'delta', 'partitioned': True}, message='switch to delta')

all_wfs = await store.list_workflows()
print('All versioned workflows:', all_wfs)
for name in all_wfs:
    wf_temp = VersionedWorkflow(name, store=store)
    latest = await wf_temp.current()
    print(f'  {name}: latest={latest.version} by {latest.author}')

## Summary

- History is **append-only** — rollbacks create a new version, nothing is deleted
- `SQLiteVersionStore` survives restarts; `InMemoryVersionStore` for tests
- `diff()` works on locally-cached versions; use `await wf.get(v)` to populate cache
- Tags let you mark `stable`, `canary`, `deprecated` etc. on any version